In [18]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Make project root importable and set as reference point for paths
sys.path.append("..")

# Build the path relative to the project root, not the notebook's location
DATA_PATH = os.path.join("..", "data", "raw", "mental-heath-in-tech-2016_20161114.csv")

df = pd.read_csv(DATA_PATH)

In [2]:
import sys
sys.path.append("..")

from src import feature_config
from src.preprocessing import run_pipeline

df = pd.read_csv(DATA_PATH)
df_processed = run_pipeline(df, feature_config)

In [3]:
df_processed.shape

(1433, 159)

In [5]:
df_processed.isnull().sum().sort_values(ascending=False).head(20)

negative_impact_reveal                        1289
negative_impact_reveal_coworker               1279
reveal_to_coworkers_direction                 1257
reveal_to_clients_direction                   1247
percentage_affected                           1229
productivity_affected                         1177
tech_role                                     1170
medical_coverage                              1146
awareness_resources                           1146
previous_employers_anonymity_protected        1029
anonymity_protected                           1029
unsupportive_response_mental_health_basis     1002
employer_takes_mental_health_seriously         780
less_likely_reveal_mental_health               776
awareness_previous_employers                   751
mental_health_resources                        607
team_view_mental_health_basis                  591
career_impact_mental_health_basis              588
interferes_with_work_treated                   557
previous_employers_mental_healt

In [11]:
df['What country do you live in?'].value_counts().head(20)

What country do you live in?
United States of America    840
United Kingdom              180
Canada                       78
Germany                      58
Netherlands                  48
Australia                    35
Sweden                       19
France                       16
Ireland                      15
Brazil                       10
Switzerland                  10
Russia                        9
India                         9
New Zealand                   9
Denmark                       7
Finland                       7
Bulgaria                      7
Belgium                       5
Italy                         5
Poland                        4
Name: count, dtype: int64

In [3]:
print([c for c in df_processed.columns if any(ch in c for ch in " '/()")])

[]


In [7]:
import src.preprocessing as pp
df_test = pp.rename_columns(pp.normalize_raw_column_names(df), feature_config.COLUMN_RENAME_MAP)

for col in feature_config.MULTISELECT_COLUMNS:
    all_values = set()
    for cell in df_test[col].dropna():
        all_values.update(v.strip() for v in str(cell).split("|"))
    print(col, "->", len(all_values), "distinct values")
    print(sorted(all_values))
    print()

diagnosed_conditions -> 33 distinct values
['ADD (w/o Hyperactivity)', 'Addictive Disorder', 'Anxiety Disorder (Generalized, Social, Phobia, etc)', 'Asperges', 'Attention Deficit Hyperactivity Disorder', 'Autism', "Autism (Asperger's)", 'Autism Spectrum Disorder', 'Autism spectrum disorder', 'Burn out', 'Combination of physical impairment (strongly near-sighted) with a possibly mental one (MCD / "ADHD", though its actually a stimulus filtering impairment)', 'Depression', 'Dissociative Disorder', 'Eating Disorder (Anorexia, Bulimia, etc)', 'Gender Dysphoria', "I haven't been formally diagnosed, so I felt uncomfortable answering, but Social Anxiety and Depression.", 'Intimate Disorder', 'Mood Disorder (Depression, Bipolar Disorder, etc)', 'Obsessive-Compulsive Disorder', 'PDD-NOS', 'PTSD (undiagnosed)', 'Personality Disorder (Borderline, Antisocial, Paranoid, etc)', 'Pervasive Developmental Disorder (Not Otherwise Specified)', 'Post-traumatic Stress Disorder', 'Psychotic Disorder (Schizo

In [4]:
import src.preprocessing as pp
df_test = pp.canonicalize_multiselect_values(
    pp.rename_columns(pp.normalize_raw_column_names(df), feature_config.COLUMN_RENAME_MAP),
    feature_config.MULTISELECT_CANONICALIZATION
)
print(df_test['diagnosed_conditions'].dropna().unique())

<StringArray>
[                                                                                                                                                                                   'Anxiety Disorder|Mood Disorder',
                                                                                                                                                                              'Anxiety Disorder|Adjustment Disorder',
                                                                                                                                                           'Anxiety Disorder|Adjustment Disorder|Addictive Disorder',
                                                                                                                                                                               'Anxiety Disorder|Addictive Disorder',
                                                                                                                                  

In [5]:
all_values = set()
for cell in df_test['diagnosed_conditions'].dropna():
    all_values.update(cell.split('|'))
print(len(all_values), sorted(all_values))

17 ['Addictive Disorder', 'Adjustment Disorder', 'Anxiety Disorder', 'Attention Deficit Hyperactivity Disorder', 'Autism', 'Burn out', 'Depression', 'Dissociative Disorder', 'Eating Disorder', 'Gender Dysphoria', 'Mood Disorder', 'Obsessive-Compulsive Disorder', 'Other', 'Personality Disorder', 'Post-traumatic Stress Disorder', 'Psychotic Disorder', 'Sleeping Disorder']


In [8]:
from src.preprocessing import (
    normalize_raw_column_names,
    rename_columns,
    canonicalize_multiselect_values,
    split_multiselect,
)

In [9]:
df_conditions_test = canonicalize_multiselect_values(
    rename_columns(normalize_raw_column_names(df), feature_config.COLUMN_RENAME_MAP),
    feature_config.MULTISELECT_CANONICALIZATION
)

df_conditions_split = split_multiselect(
    df_conditions_test,
    ["diagnosed_conditions", "believed_conditions", "diagnosed_conditions_professional"]
)

for col_prefix in ["diagnosed_conditions", "believed_conditions", "diagnosed_conditions_professional"]:
    flag_cols = [c for c in df_conditions_split.columns if c.startswith(f"{col_prefix}__")]
    counts = df_conditions_split[flag_cols].sum().sort_values(ascending=False)
    print(f"--- {col_prefix} ---")
    print(counts)
    print()

--- diagnosed_conditions ---
diagnosed_conditions__mood_disorder                               412
diagnosed_conditions__anxiety_disorder                            346
diagnosed_conditions__attention_deficit_hyperactivity_disorder    123
diagnosed_conditions__post_traumatic_stress_disorder               70
diagnosed_conditions__addictive_disorder                           60
diagnosed_conditions__obsessive_compulsive_disorder                45
diagnosed_conditions__adjustment_disorder                          38
diagnosed_conditions__personality_disorder                         36
diagnosed_conditions__eating_disorder                              28
diagnosed_conditions__dissociative_disorder                        11
diagnosed_conditions__autism                                        9
diagnosed_conditions__psychotic_disorder                            6
diagnosed_conditions__depression                                    2
diagnosed_conditions__other                                  